In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

df_bronze_tc = spark.table("bronze.tipo_cambio")

# 1. Creamos un calendario maestro con todos los días de 2005
df_calendario = spark.sql("SELECT explode(sequence(to_date('2005-01-01'), to_date('2005-12-31'))) as fecha")

# 2. Hacemos left join para que los fines de semana existan (quedarán en null)
df_tc_completo = df_calendario.join(df_bronze_tc, on="fecha", how="left")

# 3. Forward-fill: arrastra el viernes hacia el sábado/domingo
ventana_ffill = Window.orderBy("fecha").rowsBetween(Window.unboundedPreceding, 0)
df_tc_ffill = df_tc_completo.withColumn(
    "tasa_cambio_rellena", F.last("tasa_cambio", ignorenulls=True).over(ventana_ffill)
)

# 4. Back-fill (Rescate): Como el año empezó en sábado, el 1 y 2 de enero no tienen 
# viernes previo. Arrastramos la tasa del lunes (3 de enero) hacia atrás solo para esos dos días.
ventana_bfill = Window.orderBy("fecha").rowsBetween(0, Window.unboundedFollowing)
df_tc_final = df_tc_ffill.withColumn(
    "tasa_cambio_rellena", 
    F.coalesce(F.col("tasa_cambio_rellena"), F.first("tasa_cambio_rellena", ignorenulls=True).over(ventana_bfill))
)

df_tc_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tipo_cambio")

print(f"Filas del calendario maestro: {df_tc_final.count()}")
print(f"Filas sin tasa después del rescate: {df_tc_final.filter(F.col('tasa_cambio_rellena').isNull()).count()}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Filas del calendario maestro: 365
Filas sin tasa después del rescate: 0


In [0]:
df_silver_sunat = spark.table("silver.sunat_paises")
df_tc_relleno = spark.table("silver.tipo_cambio")

def calcular_equivalente_soles(df_comercio, df_tipo_cambio):
    # 🛡️ PROTECCIÓN DE IDEMPOTENCIA: 
    # Quitamos las columnas si ya existen de una ejecución anterior
    cols_a_borrar = [c for c in ["tasa_cambio_rellena", "valor_fob_pen"] if c in df_comercio.columns]
    if cols_a_borrar:
        df_comercio = df_comercio.drop(*cols_a_borrar)

    # Ahora sí, hacemos el join limpios
    df_join = df_comercio.join(
        df_tipo_cambio.select("fecha", "tasa_cambio_rellena"),
        on="fecha", 
        how="left"
    )
    
    df_resultado = df_join.withColumn(
        "valor_fob_pen", F.col("valor_fob_usd") * F.col("tasa_cambio_rellena")
    )

    # Control Duro: Abortar si algún registro se quedó sin conversión
    sin_tasa = df_resultado.filter(F.col("tasa_cambio_rellena").isNull()).count()
    assert sin_tasa == 0, f"ERROR DE AUDITORÍA: {sin_tasa} registros quedaron sin tasa de cambio asignada"

    return df_resultado

df_sunat_con_pen = calcular_equivalente_soles(df_silver_sunat, df_tc_relleno)

# Sobreescribimos la tabla en Silver
df_sunat_con_pen.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver.sunat_paises")

display(df_sunat_con_pen.select("pais_destino", "fecha", "valor_fob_usd", "tasa_cambio_rellena", "valor_fob_pen").limit(5))

pais_destino,fecha,valor_fob_usd,tasa_cambio_rellena,valor_fob_pen
PAISES BAJOS,2005-01-01,7.575091748E10,3.287,2.4899326575675998E11
COLOMBIA,2005-01-01,2.232510903E10,3.287,7.338263338161E10
SUECIA,2005-01-01,2.17278285E9,3.287,7.14193722795E9
URUGUAY,2005-01-01,4.7060693E8,3.287,1.5468849789099998E9
EGIPTO,2005-01-01,2.5216843E8,3.287,8.2887762941E8
